# Phase 9 — Adversarial Training

## Objective

Train a neural-network IDS using both clean and adversarial
training samples and determine whether adversarial training
improves robustness against FGSM and PGD attacks.

## Defense

FGSM-based adversarial training.

## Training strategy

Each training batch contains:
- clean network-flow samples
- FGSM-perturbed samples

The validation and test datasets are not used during training.

## Evaluation

The adversarially trained model will be evaluated on:
1. Clean validation data
2. FGSM validation data
3. PGD validation data

The experiment measures both:
- clean-data performance
- adversarial robustness

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

sys.path.append("..")

from src.attacks.fgsm import fgsm_attack
from src.attacks.pgd import pgd_attack

In [2]:
SPLIT_DIR = Path("../data/splits")
MODEL_DIR = Path("../models")
RESULTS_DIR = Path("../experiments/results")

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [3]:
X_train = pd.read_csv(
    SPLIT_DIR / "X_train.csv"
)

y_train = pd.read_csv(
    SPLIT_DIR / "y_train.csv"
).squeeze()

X_val = pd.read_csv(
    SPLIT_DIR / "X_val.csv"
)

y_val = pd.read_csv(
    SPLIT_DIR / "y_val.csv"
).squeeze()

print("Training:", X_train.shape)
print("Validation:", X_val.shape)

Training: (1764558, 70)
Validation: (378120, 70)


In [4]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

X_train_tensor = torch.tensor(
    X_train.values,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train.values,
    dtype=torch.float32
).view(-1, 1)

X_val_tensor = torch.tensor(
    X_val.values,
    dtype=torch.float32,
    device=device
)

y_val_tensor = torch.tensor(
    y_val.values,
    dtype=torch.float32,
    device=device
).view(-1, 1)

print("Device:", device)

Device: cpu


In [5]:
class IDSNeuralNetwork(nn.Module):

    def __init__(self, input_size):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)

In [6]:
input_size = X_train.shape[1]

adv_model = IDSNeuralNetwork(
    input_size
).to(device)

print(adv_model)

IDSNeuralNetwork(
  (network): Sequential(
    (0): Linear(in_features=70, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=32, bias=True)
    (5): ReLU()
    (6): Linear(in_features=32, out_features=1, bias=True)
  )
)


In [7]:
loss_fn = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    adv_model.parameters(),
    lr=0.001
)

In [8]:
batch_size = 1024

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

In [9]:
ADV_EPSILON = 0.01
EPOCHS = 5

In [10]:
ADV_EPSILON = 0.01
EPOCHS = 5

for epoch in range(EPOCHS):

    adv_model.train()

    total_loss = 0.0
    total_clean_loss = 0.0
    total_adv_loss = 0.0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # ------------------------------------------------
        # 1. Generate adversarial training examples
        # ------------------------------------------------

        X_adv_batch = fgsm_attack(
            model=adv_model,
            x=X_batch,
            y=y_batch,
            epsilon=ADV_EPSILON,
            loss_fn=loss_fn
        )

        # ------------------------------------------------
        # 2. Clean loss
        # ------------------------------------------------

        clean_logits = adv_model(
            X_batch
        )

        clean_loss = loss_fn(
            clean_logits,
            y_batch
        )

        # ------------------------------------------------
        # 3. Adversarial loss
        # ------------------------------------------------

        adv_logits = adv_model(
            X_adv_batch
        )

        adv_loss = loss_fn(
            adv_logits,
            y_batch
        )

        # ------------------------------------------------
        # 4. Combined loss
        # ------------------------------------------------

        loss = (
            0.5 * clean_loss +
            0.5 * adv_loss
        )

        # ------------------------------------------------
        # 5. Backpropagation
        # ------------------------------------------------

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        # ------------------------------------------------
        # 6. Track losses
        # ------------------------------------------------

        total_loss += loss.item()

        total_clean_loss += (
            clean_loss.item()
        )

        total_adv_loss += (
            adv_loss.item()
        )

    num_batches = len(train_loader)

    print(
        f"Epoch [{epoch + 1}/{EPOCHS}] "
        f"Total Loss: {total_loss / num_batches:.4f} "
        f"Clean Loss: {total_clean_loss / num_batches:.4f} "
        f"Adv Loss: {total_adv_loss / num_batches:.4f}"
    )

Epoch [1/5] Total Loss: 0.0730 Clean Loss: 0.0674 Adv Loss: 0.0786
Epoch [2/5] Total Loss: 0.0485 Clean Loss: 0.0433 Adv Loss: 0.0537
Epoch [3/5] Total Loss: 0.0447 Clean Loss: 0.0402 Adv Loss: 0.0493
Epoch [4/5] Total Loss: 0.0427 Clean Loss: 0.0386 Adv Loss: 0.0467
Epoch [5/5] Total Loss: 0.0418 Clean Loss: 0.0381 Adv Loss: 0.0454


In [11]:
ADV_MODEL_PATH = (
    MODEL_DIR /
    "neural_network_adversarial_fgsm.pth"
)

torch.save(
    adv_model.state_dict(),
    ADV_MODEL_PATH
)

print(
    "Saved:",
    ADV_MODEL_PATH
)

Saved: ..\models\neural_network_adversarial_fgsm.pth


In [12]:
def calculate_metrics(
    y_true,
    y_pred,
    y_prob
):

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred
    ).ravel()

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y_true,
        y_prob
    )

    pr_auc = average_precision_score(
        y_true,
        y_prob
    )

    fpr = fp / (fp + tn)

    fnr = fn / (fn + tp)

    return {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC-AUC": roc_auc,
        "PR-AUC": pr_auc,
        "FPR": fpr,
        "FNR": fnr,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    }

In [13]:
def calculate_metrics(
    y_true,
    y_pred,
    y_prob
):

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred
    ).ravel()

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y_true,
        y_prob
    )

    pr_auc = average_precision_score(
        y_true,
        y_prob
    )

    fpr = fp / (fp + tn)

    fnr = fn / (fn + tp)

    return {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC-AUC": roc_auc,
        "PR-AUC": pr_auc,
        "FPR": fpr,
        "FNR": fnr,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    }

In [14]:
adv_model.eval()

with torch.no_grad():

    clean_logits = adv_model(
        X_val_tensor
    )

    clean_prob = torch.sigmoid(
        clean_logits
    ).cpu().numpy().ravel()

    clean_pred = (
        clean_prob >= 0.5
    ).astype(int)

In [15]:
adv_train_clean_metrics = calculate_metrics(
    y_val.values,
    clean_pred,
    clean_prob
)

adv_train_clean_metrics

{'Accuracy': 0.9816804189146303,
 'Precision': 0.9715421567003478,
 'Recall': 0.9184322199777643,
 'F1': 0.9442409704502097,
 'ROC-AUC': 0.9984341290555641,
 'PR-AUC': 0.992876134528153,
 'FPR': np.float64(0.005466828316770562),
 'FNR': np.float64(0.08156778002223579),
 'TN': np.int64(312541),
 'FP': np.int64(1718),
 'FN': np.int64(5209),
 'TP': np.int64(58652)}

In [16]:
FGSM_EPSILON = 0.01

X_val_fgsm = fgsm_attack(
    model=adv_model,
    x=X_val_tensor,
    y=y_val_tensor,
    epsilon=FGSM_EPSILON,
    loss_fn=loss_fn
)

In [17]:
with torch.no_grad():

    fgsm_logits = adv_model(
        X_val_fgsm
    )

    fgsm_prob = torch.sigmoid(
        fgsm_logits
    ).cpu().numpy().ravel()

    fgsm_pred = (
        fgsm_prob >= 0.5
    ).astype(int)

In [18]:
adv_train_fgsm_metrics = calculate_metrics(
    y_val.values,
    fgsm_pred,
    fgsm_prob
)

adv_train_fgsm_metrics

{'Accuracy': 0.9777054903205331,
 'Precision': 0.9560645702720048,
 'Recall': 0.9098041057922676,
 'F1': 0.9323608704024648,
 'ROC-AUC': 0.9973839875529456,
 'PR-AUC': 0.9885056903626511,
 'FPR': np.float64(0.008496176720475786),
 'FNR': np.float64(0.09019589420773241),
 'TN': np.int64(311589),
 'FP': np.int64(2670),
 'FN': np.int64(5760),
 'TP': np.int64(58101)}

In [19]:
attack_mask_adv_model = (
    (y_val.values == 1) &
    (clean_pred == 1)
)

X_correct_attacks = (
    X_val_tensor[
        attack_mask_adv_model
    ]
)

y_correct_attacks = (
    y_val_tensor[
        attack_mask_adv_model
    ]
)

print(
    "Correctly detected attacks:",
    len(X_correct_attacks)
)

Correctly detected attacks: 58652


In [20]:
X_correct_attacks_fgsm = fgsm_attack(
    model=adv_model,
    x=X_correct_attacks,
    y=y_correct_attacks,
    epsilon=FGSM_EPSILON,
    loss_fn=loss_fn
)

In [21]:
with torch.no_grad():

    attack_logits = adv_model(
        X_correct_attacks_fgsm
    )

    attack_prob = torch.sigmoid(
        attack_logits
    ).cpu().numpy().ravel()

    attack_pred = (
        attack_prob >= 0.5
    ).astype(int)

In [22]:
successful_evasions = (
    attack_pred == 0
).sum()

attack_success_rate = (
    successful_evasions /
    len(attack_pred)
)

print(
    "FGSM Attack Success Rate:",
    attack_success_rate
)

FGSM Attack Success Rate: 0.00939439405305872


In [23]:
PGD_EPSILON = 0.01
PGD_STEPS = 10
PGD_ALPHA = PGD_EPSILON / PGD_STEPS

In [24]:
X_val_pgd = pgd_attack(
    model=adv_model,
    x=X_val_tensor,
    y=y_val_tensor,
    epsilon=PGD_EPSILON,
    alpha=PGD_ALPHA,
    num_steps=PGD_STEPS,
    loss_fn=loss_fn
)

In [26]:
with torch.no_grad():

    pgd_logits = adv_model(
        X_val_pgd
    )

    pgd_prob = torch.sigmoid(
        pgd_logits
    ).cpu().numpy().ravel()

    pgd_pred = (
        pgd_prob >= 0.5
    ).astype(int)

In [27]:
adv_train_pgd_metrics = calculate_metrics(
    y_val.values,
    pgd_pred,
    pgd_prob
)

adv_train_pgd_metrics

{'Accuracy': 0.9775970591346662,
 'Precision': 0.9554948850366765,
 'Recall': 0.9097258107452123,
 'F1': 0.9320488035744366,
 'ROC-AUC': 0.9971950311012254,
 'PR-AUC': 0.9877286967195287,
 'FPR': np.float64(0.008610731912212538),
 'FNR': np.float64(0.09027418925478774),
 'TN': np.int64(311553),
 'FP': np.int64(2706),
 'FN': np.int64(5765),
 'TP': np.int64(58096)}

In [28]:
X_correct_attacks_pgd = pgd_attack(
    model=adv_model,
    x=X_correct_attacks,
    y=y_correct_attacks,
    epsilon=PGD_EPSILON,
    alpha=PGD_ALPHA,
    num_steps=PGD_STEPS,
    loss_fn=loss_fn
)

In [29]:
with torch.no_grad():

    attack_pgd_logits = adv_model(
        X_correct_attacks_pgd
    )

    attack_pgd_prob = torch.sigmoid(
        attack_pgd_logits
    ).cpu().numpy().ravel()

    attack_pgd_pred = (
        attack_pgd_prob >= 0.5
    ).astype(int)

In [30]:
pgd_successful_evasions = (
    attack_pgd_pred == 0
).sum()

pgd_attack_success_rate = (
    pgd_successful_evasions /
    len(attack_pgd_pred)
)

print(
    "PGD Attack Success Rate:",
    pgd_attack_success_rate
)

PGD Attack Success Rate: 0.00947964263793221


In [32]:
defense_results = pd.DataFrame([
    {
        "Model": "Adversarially Trained NN",
        "Condition": "Clean",
        **adv_train_clean_metrics,
        "Attack Success Rate": np.nan
    },
    {
        "Model": "Adversarially Trained NN",
        "Condition": "FGSM",
        **adv_train_fgsm_metrics,
        "Attack Success Rate": attack_success_rate
    },
    {
        "Model": "Adversarially Trained NN",
        "Condition": "PGD",
        **adv_train_pgd_metrics,
        "Attack Success Rate": pgd_attack_success_rate
    }
])

defense_results

,Model,Condition,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC,FPR,FNR,TN,FP,FN,TP,Attack Success Rate
0,Adversarially Trained NN,Clean,0.981680,0.971542,0.918432,0.944241,0.998434,0.992876,0.005467,0.081568,312541,1718,5209,58652,NaN
1,Adversarially Trained NN,FGSM,0.977705,0.956065,0.909804,0.932361,0.997384,0.988506,0.008496,0.090196,311589,2670,5760,58101,0.009394
2,Adversarially Trained NN,PGD,0.977597,0.955495,0.909726,0.932049,0.997195,0.987729,0.008611,0.090274,311553,2706,5765,58096,0.009480


In [33]:
defense_results.to_csv(
    RESULTS_DIR /
    "adversarial_training_results.csv",
    index=False
)

print(
    "Adversarial training results saved."
)

Adversarial training results saved.
